# Deformed ²⁰⁸Pb+²⁰⁸Pb @ 2.76 TeV — Workstation Pipeline

**Maximally parallel across all CPU cores.  Run cells 1–9 in order.**

| Cell | Stage | Parallelism |
|------|-------|-------------|
| 1 | Configuration ← edit here | — |
| 2 | Prerequisites (auto-install) | — |
| 3 | Build `trento_sync` | `make -jN` |
| 4 | Clone Isobar-Sampler | — |
| 5 | Physics, LHS design, seed bank | all CPUs |
| 6 | Write worker module | — |
| 7 | **Parallel main loop** | `ProcessPoolExecutor` |
| 8 | Save eccentricities | — |
| 9 | All plots | — |

---
## Cell 1 — Configuration  ← **edit only this cell**

In [ ]:
import multiprocessing, os, subprocess, sys, shutil, time
from pathlib import Path

# ─── PATHS  (edit these for your workstation) ─────────────────────────────────
WORK_DIR   = Path.home() / "pbpb_deformed"   # all scan outputs
TRENTO_DIR = Path.home() / "trento_sync"      # trento_sync repo + build
ISO_DIR    = Path.home() / "Isobar-Sampler"  # ThiagoSDomingues fork

# ─── PHYSICS ──────────────────────────────────────────────────────────────────
N_DESIGN      = 50        # LHS design points
N_CONFIGS     = 1_000     # nuclear configs per design pt  (Isobar Sampler)
N_EVENTS      = 100_000   # TRENTo events per design pt
N_LHS_ITER    = 2000      # maximin LHS trials
SEED_LHS      = 42        # RNG seed  (identical to Colab version for comparison)

# ─── PARALLELISM  ← tune for your machine ─────────────────────────────────────
N_CORES         = multiprocessing.cpu_count()   # all logical CPUs
N_ISO_PER_JOB   = 4   # isobar sampler threads per design-point job
                       # ↑ 4 is good for ≤16-core machines
                       # ↑ 8 recommended for 32+ cores
N_OUTER_WORKERS = max(1, N_CORES // N_ISO_PER_JOB)  # simultaneous design pts

print(f"CPUs detected      : {N_CORES}")
print(f"ISO cores / job    : {N_ISO_PER_JOB}")
print(f"Outer workers      : {N_OUTER_WORKERS}  (design pts run in parallel)")
print(f"Effective CPU use  : ~{N_OUTER_WORKERS*(N_ISO_PER_JOB+1)} / {N_CORES}")
print(f"N_DESIGN           : {N_DESIGN}")
print(f"N_EVENTS / pt      : {N_EVENTS:,}")
print(f"N_CONFIGS / pt     : {N_CONFIGS:,}")


---
## Cell 2 — Prerequisites

In [ ]:
import platform, importlib

def _run(cmd, check=True, **kw):
    cmd = [str(c) for c in cmd]
    print("  $ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, check=False, **kw)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited {r.returncode}")
    return r

print("=" * 60)
print("  Prerequisites")
print("=" * 60)

# ── Python packages ────────────────────────────────────────────────────────────
print("\n--- Python packages ---")
_run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
      "numpy", "scipy", "matplotlib", "h5py", "pyyaml", "joblib", "tqdm"])
print("  OK")

# ── Detect package manager ─────────────────────────────────────────────────────
PKG_MGR = next((p for p in ["apt-get","dnf","yum","pacman","brew"]
                if shutil.which(p)), None)
OS = platform.system()
print(f"\n--- System  ({OS} | {PKG_MGR or 'no pkg mgr'}) ---")

def _sys(*pkgs):
    if PKG_MGR is None:
        raise RuntimeError(
            f"No supported package manager found.\n"
            f"Please install manually: {pkgs}"
        )
    if PKG_MGR == "apt-get":
        subprocess.run(["sudo","apt-get","update","-qq"], check=False)
        _run(["sudo","apt-get","install","-y","-qq",*pkgs])
    elif PKG_MGR in ("dnf","yum"):
        _run(["sudo", PKG_MGR, "install","-y",*pkgs])
    elif PKG_MGR == "pacman":
        _run(["sudo","pacman","-S","--noconfirm",*pkgs])
    elif PKG_MGR == "brew":
        _run(["brew","install",*pkgs])

# cmake
if not shutil.which("cmake"):
    print("  cmake not found — installing...")
    _sys("cmake")
else:
    v = subprocess.run(["cmake","--version"],capture_output=True,text=True)
    print(f"  cmake   : {v.stdout.splitlines()[0]}")

# g++
cxx = shutil.which("g++") or shutil.which("c++")
if not cxx:
    print("  g++ not found — installing...")
    if PKG_MGR == "apt-get":
        _sys("build-essential")
    elif PKG_MGR in ("dnf","yum"):
        _sys("gcc-c++","make")
    elif PKG_MGR == "pacman":
        _sys("base-devel")
    elif PKG_MGR == "brew":
        _sys("gcc")
    else:
        raise RuntimeError("Install g++ manually.")
else:
    v = subprocess.run([cxx,"--version"],capture_output=True,text=True)
    print(f"  g++     : {v.stdout.splitlines()[0]}")

# Boost headers
BOOST_PATHS = [
    "/usr/include/boost/version.hpp",
    "/usr/local/include/boost/version.hpp",
    "/opt/homebrew/include/boost/version.hpp",
]
if not any(Path(p).exists() for p in BOOST_PATHS):
    print("  Boost not found — installing...")
    if PKG_MGR == "apt-get":      _sys("libboost-all-dev")
    elif PKG_MGR in ("dnf","yum"):_sys("boost-devel")
    elif PKG_MGR == "pacman":     _sys("boost")
    elif PKG_MGR == "brew":       _sys("boost")
    else: raise RuntimeError("Install boost-devel / libboost-all-dev manually.")
else:
    print("  Boost   : found")

# git
if not shutil.which("git"):
    print("  git not found — installing...")
    _sys("git") if PKG_MGR != "brew" else _sys("git")
else:
    v = subprocess.run(["git","--version"],capture_output=True,text=True)
    print(f"  git     : {v.stdout.strip()}")

# h5py / HDF5
try:
    import h5py
    print(f"  h5py    : {h5py.__version__}  (HDF5 {h5py.version.hdf5_version})")
except ImportError:
    _run([sys.executable,"-m","pip","install","-q","h5py"])
    import h5py
    print(f"  h5py    : {h5py.__version__}")

print("\n=== All prerequisites satisfied ===")


---
## Cell 3 — Build trento_sync

In [ ]:
TRENTO_BUILD = TRENTO_DIR / "build"

def _find_trento():
    candidates = [
        "/usr/local/bin/trento",
        "/usr/bin/trento",
        str(TRENTO_BUILD / "src" / "trento"),
        str(TRENTO_BUILD / "trento"),
        str(Path.home() / ".local" / "bin" / "trento"),
    ]
    for c in candidates:
        if os.path.isfile(c) and os.access(c, os.X_OK):
            return Path(c)
    r = subprocess.run(["which","trento"], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        return Path(r.stdout.strip())
    try:
        r2 = subprocess.run(
            ["find", str(Path.home()), "-name", "trento",
             "-type", "f", "-perm", "/111"],
            capture_output=True, text=True, timeout=15,
        )
        for line in r2.stdout.strip().splitlines():
            line = line.strip()
            if line and os.path.isfile(line) and os.access(line, os.X_OK):
                return Path(line)
    except Exception:
        pass
    return None

print("=" * 60)
print("  Stage 1 — trento_sync")
print("=" * 60)

TRENTO_BIN = _find_trento()

if TRENTO_BIN:
    print(f"  Already built → {TRENTO_BIN}")
else:
    print(f"  Building on {N_CORES} cores…")
    if TRENTO_DIR.exists():
        shutil.rmtree(TRENTO_DIR)
    _run(["git", "clone", "--depth=1",
          "https://github.com/jppicchetti/trento_sync.git",
          str(TRENTO_DIR)])
    TRENTO_BUILD.mkdir(parents=True, exist_ok=True)
    # cmake: source dir as positional arg, Release for full optimisation
    _run(["cmake", str(TRENTO_DIR), "-DCMAKE_BUILD_TYPE=Release"],
         cwd=TRENTO_BUILD)
    t0 = time.perf_counter()
    _run(["make", f"-j{N_CORES}"], cwd=TRENTO_BUILD)
    print(f"  Compiled in {time.perf_counter()-t0:.0f} s on {N_CORES} cores")
    subprocess.run(["make", "install"], cwd=str(TRENTO_BUILD), check=False)
    TRENTO_BIN = _find_trento()
    if not TRENTO_BIN:
        raise RuntimeError("trento_sync binary not found after build")

ver = subprocess.run([str(TRENTO_BIN), "--version"],
                     capture_output=True, text=True)
print(f"  Version → {(ver.stdout+ver.stderr).strip().splitlines()[0]}")
print(f"  Binary  → {TRENTO_BIN}")


---
## Cell 4 — Clone Isobar-Sampler

In [ ]:
print("=" * 60)
print("  Stage 2 — Isobar Sampler")
print("=" * 60)

if ISO_DIR.exists() and (ISO_DIR / "exec" / "make_seeds.py").exists():
    print(f"  Already at {ISO_DIR} — skipping clone")
else:
    if ISO_DIR.exists():
        shutil.rmtree(ISO_DIR)
    _run(["git", "clone", "--depth=1",
          "https://github.com/ThiagoSDomingues/Isobar-Sampler",
          str(ISO_DIR)])

print(f"  make_seeds.py    ✓")
print(f"  build_isobars.py ✓")


---
## Cell 5 — Physics, LHS Design, Seed Bank

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import yaml
import warnings
warnings.filterwarnings("ignore")

print("=" * 60)
print("  Stage 3 — Physics constants, LHS design, Seed bank")
print("=" * 60)

# ── 5 FREE parameters (LHS-sampled) ───────────────────────────────────────────
PARAMS = [
    # name     lo      hi      latex             stage
    ("R",      6.50,   6.80,   r"$R$ [fm]",     "IS"),
    ("a",      0.44,   0.65,   r"$a$ [fm]",     "IS"),
    ("beta3",  0.00,   0.12,   r"$\beta_3$",   "IS"),
    ("beta4", -0.02,   0.06,   r"$\beta_4$",   "IS"),
    ("w",      0.40,   1.20,   r"$w$ [fm]",     "TR"),
]
PNAMES  = [p[0] for p in PARAMS]
PLO     = np.array([p[1] for p in PARAMS])
PHI     = np.array([p[2] for p in PARAMS])
PLABELS = [p[3] for p in PARAMS]
N_PAR   = len(PARAMS)

# ── Fixed TRENTo params (JETSCAPE Grad MAP, arXiv:2011.01430 Table VII) ────────
FIX_P    = 0.063
FIX_K    = 1.0 / 0.97**2    # sigma_k=0.97  →  k ≈ 1.063
FIX_NORM = 18.12
FIX_DMIN = 0.52**(1.0/3.0)  # d³=0.52 fm³  →  d_min=0.804 fm
SIGMA_NN = 6.4
B_MAX    = 20.0

# ── Fixed Isobar Sampler params ────────────────────────────────────────────────
FIX_BETA2  = 0.0   # 208Pb doubly magic
FIX_GAMMA  = 0.0
FIX_CORR_L = 0.4   # correlation length [fm]
FIX_CORR_S = -1.0  # full Pauli exclusion
A_PB       = 208

# ── trento_sync column indices (13 cols vs 8 in stock TRENTo) ─────────────────
# 0:ev  1:b  2:npart  3:mult_tot  4:mult_mid
# 5:|ε₂| 6:Ψ₂  7:|ε₃|  8:Ψ₃  9:|ε₄|  10:Ψ₄  11:|ε₅|  12:Ψ₅
N_SYNC_COLS = 13
COL_MULT    = 3   # mult_tot → centrality
COL_E2      = 5   # |ε₂|
COL_E3      = 7   # |ε₃|
COL_E4      = 9   # |ε₄|
COL_E5      = 11  # |ε₅|

print(f"  p={FIX_P}  k={FIX_K:.4f}  norm={FIX_NORM}  d_min={FIX_DMIN:.4f} fm")

def write_yaml(path, data):
    with open(path, "w") as fh:
        yaml.dump(data, fh, default_flow_style=False, sort_keys=False)

def fmt(s):
    if s < 60:   return f"{s:.1f} s"
    if s < 3600: return f"{s/60:.1f} min"
    return f"{s/3600:.2f} h"

# ── Eccentricity helpers (also used to load cached results in Cell 7) ──────────
def assign_centrality(events):
    rank = np.argsort(np.argsort(-events[:, COL_MULT]))
    return rank / len(events) * 100.0

def _m(eps, order): return np.mean(eps**order)

def ecc2_cumulants(eps):
    m2    = _m(eps, 2)
    m4    = _m(eps, 4)
    inner = 2.0 * m2**2 - m4
    return np.sqrt(m2), (inner**0.25 if inner >= 0 else np.nan)

def compute_observables(events, centrality, c_lo, c_hi):
    mask = (centrality >= c_lo) & (centrality < c_hi)
    sel  = events[mask]
    if len(sel) < 10:
        d = {k: np.nan for k in
             ["e2_2","e3_2","e4_2","T32","R42","e3_ratio","e4_4c"]}
        d["n"] = 0
        return d
    e2_2, e2_4 = ecc2_cumulants(sel[:, COL_E2])
    e3_2, e3_4 = ecc2_cumulants(sel[:, COL_E3])
    e4_2, _    = ecc2_cumulants(sel[:, COL_E4])
    T32   = e3_2 / e2_2 if e2_2 > 0 else np.nan
    R42   = e2_4 / e2_2 if (e2_2 > 0 and not np.isnan(e2_4)) else np.nan
    e3rat = e3_2 / e3_4 if (e3_4 > 0 and not np.isnan(e3_4)) else np.nan
    e4_4c = 2.0 * _m(sel[:, COL_E4], 2)**2 - _m(sel[:, COL_E4], 4)
    return dict(e2_2=e2_2, e3_2=e3_2, e4_2=e4_2,
                T32=T32, R42=R42, e3_ratio=e3rat, e4_4c=e4_4c,
                n=int(mask.sum()))

def compute_centrality_profiles(events, centrality):
    uc_e   = list(range(11))
    wide_e = [0,5,10,20,30,40,50,60,70,80,90]
    keys   = ["e2_2","e3_2","e4_2","T32","R42","e3_ratio","e4_4c"]
    result = {}
    for tag, edges in [("uc", uc_e), ("wide", wide_e)]:
        bins = list(zip(edges[:-1], edges[1:]))
        mid  = np.array([(lo+hi)/2.0 for lo, hi in bins])
        data = {k: [] for k in keys}
        ns   = []
        for lo, hi in bins:
            obs = compute_observables(events, centrality, lo, hi)
            for k in keys:
                data[k].append(obs[k])
            ns.append(obs["n"])
        result[tag] = dict(mid=mid, bins=bins, n=np.array(ns),
                           **{k: np.array(data[k]) for k in keys})
    return result

CENT_CLASSES = [("0-1",0,1),("0-5",0,5),("5-10",5,10),
                ("0-10",0,10),("20-30",20,30),("40-50",40,50)]
OBS_KEYS = ["e2_2","e3_2","e4_2","T32","R42","e3_ratio","e4_4c"]

# ── LHS design ─────────────────────────────────────────────────────────────────
WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "plots").mkdir(exist_ok=True)

lhs_path = WORK_DIR / "lhs_design.npz"
if lhs_path.exists():
    d        = np.load(lhs_path, allow_pickle=True)
    unit_lhs = d["unit_lhs"]
    design   = d["design"]
    print(f"  LHS loaded from cache  {design.shape}")
else:
    def _one_lhs(n, d, rng):
        X = np.empty((n, d))
        for j in range(d):
            perm = rng.permutation(n)
            X[:, j] = (perm + rng.uniform(size=n)) / n
        return X

    def _maximin_lhs(n, d, n_iter, seed):
        rng = np.random.default_rng(seed)
        best, bd = None, -1.0
        for _ in range(n_iter):
            X    = _one_lhs(n, d, rng)
            diff = X[:, None] - X[None]
            dist = np.sqrt((diff**2).sum(-1))
            np.fill_diagonal(dist, np.inf)
            dm = dist.min()
            if dm > bd:
                bd, best = dm, X.copy()
        return best

    print(f"  Generating {N_DESIGN}-pt maximin LHS ({N_LHS_ITER} trials)…",
          end="", flush=True)
    t0       = time.perf_counter()
    unit_lhs = _maximin_lhs(N_DESIGN, N_PAR, N_LHS_ITER, SEED_LHS)
    design   = PLO + unit_lhs * (PHI - PLO)
    print(f" done ({fmt(time.perf_counter()-t0)})")
    np.savez_compressed(lhs_path, design=design, unit_lhs=unit_lhs,
                        param_names=np.array(PNAMES), lo=PLO, hi=PHI)
    print(f"  Saved → {lhs_path}")

print(f"\n  {'idx':>4}  {'R':>6}  {'a':>6}  {'β₃':>8}  {'β₄':>8}  {'w':>6}")
print("  " + "-"*46)
for i in range(min(5, N_DESIGN)):
    r = design[i]
    print(f"  {i:4d}  {r[0]:6.4f}  {r[1]:6.4f}  {r[2]:8.5f}  {r[3]:8.5f}  {r[4]:6.4f}")
print(f"  … ({N_DESIGN} total)")

# ── Seed bank (runs ONCE, shared across all design pts) ───────────────────────
seeds_dir  = WORK_DIR / "seeds"
seeds_dir.mkdir(exist_ok=True)
seeds_hdf  = seeds_dir / "nucleon-seeds.hdf"
seeds_conf = seeds_dir / "seeds-conf.yaml"

if seeds_hdf.exists() and seeds_hdf.stat().st_size > 0:
    print(f"\n  Seed bank exists ({seeds_hdf.stat().st_size/1e6:.1f} MB) — skip")
else:
    write_yaml(seeds_conf, {
        "isobar_seeds": {
            "description": f"208Pb seeds, {N_CONFIGS} configs.",
            "number_nucleons":  {"description": "A",    "value": A_PB},
            "number_configs":   {"description": "N",    "value": N_CONFIGS},
            "output_file":      {"description": "HDF5",
                                 "filename": str(seeds_hdf.resolve())},
            "number_of_parallel_processes":
                                {"description": "cores","value": N_CORES},
        }
    })
    print(f"\n  Running make_seeds.py  ({N_CONFIGS:,} configs, {N_CORES} CPUs)…")
    t0 = time.perf_counter()
    subprocess.run([sys.executable,
                    str(ISO_DIR / "exec" / "make_seeds.py"),
                    str(seeds_conf)],
                   cwd=str(ISO_DIR), check=True)
    t_s = time.perf_counter() - t0
    if not seeds_hdf.exists():
        raise FileNotFoundError(f"Seeds not found: {seeds_hdf}")
    print(f"  Done  {seeds_hdf.stat().st_size/1e6:.1f} MB  ({fmt(t_s)})")


---
## Cell 6 — Write Worker Module

Writes `_worker_template.py` to `WORK_DIR`, then instantiates `_pbpb_worker.py` with runtime constants injected.
The worker module must be a **top-level importable file** so `ProcessPoolExecutor` can pickle and import it in worker processes.

In [ ]:
# Write worker template to WORK_DIR so it can be imported by ProcessPoolExecutor.
# The template is stored as a .py file to avoid all string-escaping issues.

import ast

_template_lines = [
    '# _pbpb_worker.py  — parallel design-point worker',
    '# Auto-generated by the notebook (Cell 6). Do not edit.',
    '# Constants are injected by Cell 6 at runtime via string substitution.',
    'import subprocess, sys, shutil, time, os',
    'import numpy as np',
    'import yaml',
    'from pathlib import Path',
    '',
    '# ── injected constants ────────────────────────────────────────────────────────',
    'TRENTO_BIN    = Path("INJECT_TRENTO_BIN")',
    'ISO_DIR       = Path("INJECT_ISO_DIR")',
    'WORK_DIR      = Path("INJECT_WORK_DIR")',
    'SEEDS_HDF     = Path("INJECT_SEEDS_HDF")',
    'N_CONFIGS     = INJECT_N_CONFIGS',
    'N_EVENTS      = INJECT_N_EVENTS',
    'N_ISO_PER_JOB = INJECT_N_ISO_PER_JOB',
    'A_PB          = INJECT_A_PB',
    'FIX_P         = INJECT_FIX_P',
    'FIX_K         = INJECT_FIX_K',
    'FIX_NORM      = INJECT_FIX_NORM',
    'FIX_DMIN      = INJECT_FIX_DMIN',
    'SIGMA_NN      = INJECT_SIGMA_NN',
    'B_MAX         = INJECT_B_MAX',
    'FIX_BETA2     = INJECT_FIX_BETA2',
    'FIX_GAMMA     = INJECT_FIX_GAMMA',
    'FIX_CORR_L    = INJECT_FIX_CORR_L',
    'FIX_CORR_S    = INJECT_FIX_CORR_S',
    'N_SYNC_COLS   = INJECT_N_SYNC_COLS',
    'COL_MULT      = INJECT_COL_MULT',
    'COL_E2        = INJECT_COL_E2',
    'COL_E3        = INJECT_COL_E3',
    'COL_E4        = INJECT_COL_E4',
    '',
    'def fmt(s):',
    '    if s < 60:   return "{:.1f} s".format(s)',
    '    if s < 3600: return "{:.1f} min".format(s/60)',
    '    return "{:.2f} h".format(s/3600)',
    '',
    'def write_yaml(path, data):',
    '    with open(path, "w") as fh:',
    '        yaml.dump(data, fh, default_flow_style=False, sort_keys=False)',
    '',
    'def _isobar_block(name, R, a, beta3, beta4):',
    '    return {',
    '        "isobar_name": name,',
    '        "WS_radius":            {"description": "R",  "value": float(R)},',
    '        "WS_diffusiveness":     {"description": "a",  "value": float(a)},',
    '        "beta_2":               {"description": "b2", "value": FIX_BETA2},',
    '        "gamma":                {"description": "g",  "value": FIX_GAMMA},',
    '        "beta_3":               {"description": "b3", "value": float(beta3)},',
    '        "correlation_length":   {"description": "Cl", "value": FIX_CORR_L},',
    '        "correlation_strength": {"description": "Cs", "value": FIX_CORR_S},',
    '        "beta_4":               {"description": "b4", "value": float(beta4)},',
    '    }',
    '',
    'def _m(eps, order):',
    '    return np.mean(eps ** order)',
    '',
    'def ecc2_cumulants(eps):',
    '    m2    = _m(eps, 2)',
    '    m4    = _m(eps, 4)',
    '    inner = 2.0 * m2**2 - m4',
    '    return np.sqrt(m2), (inner**0.25 if inner >= 0 else float("nan"))',
    '',
    'def assign_centrality(events):',
    '    rank = np.argsort(np.argsort(-events[:, COL_MULT]))',
    '    return rank / len(events) * 100.0',
    '',
    'def compute_observables(events, centrality, c_lo, c_hi):',
    '    mask = (centrality >= c_lo) & (centrality < c_hi)',
    '    sel  = events[mask]',
    '    if len(sel) < 10:',
    '        d = {k: float("nan") for k in',
    '             ["e2_2","e3_2","e4_2","T32","R42","e3_ratio","e4_4c"]}',
    '        d["n"] = 0',
    '        return d',
    '    e2_2, e2_4 = ecc2_cumulants(sel[:, COL_E2])',
    '    e3_2, e3_4 = ecc2_cumulants(sel[:, COL_E3])',
    '    e4_2, _    = ecc2_cumulants(sel[:, COL_E4])',
    '    T32   = e3_2 / e2_2  if e2_2 > 0 else float("nan")',
    '    R42   = e2_4 / e2_2  if (e2_2 > 0 and e2_4 == e2_4) else float("nan")',
    '    e3rat = e3_2 / e3_4  if (e3_4 > 0 and e3_4 == e3_4) else float("nan")',
    '    e4_4c = 2.0 * _m(sel[:, COL_E4], 2)**2 - _m(sel[:, COL_E4], 4)',
    '    return dict(e2_2=e2_2, e3_2=e3_2, e4_2=e4_2,',
    '                T32=T32, R42=R42, e3_ratio=e3rat, e4_4c=e4_4c,',
    '                n=int(mask.sum()))',
    '',
    'def compute_centrality_profiles(events, centrality):',
    '    uc_edges   = list(range(11))',
    '    wide_edges = [0, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90]',
    '    obs_keys   = ["e2_2","e3_2","e4_2","T32","R42","e3_ratio","e4_4c"]',
    '    result = {}',
    '    for tag, edges in [("uc", uc_edges), ("wide", wide_edges)]:',
    '        bins = list(zip(edges[:-1], edges[1:]))',
    '        mid  = np.array([(lo+hi)/2.0 for lo, hi in bins])',
    '        data = {k: [] for k in obs_keys}',
    '        ns   = []',
    '        for lo, hi in bins:',
    '            obs = compute_observables(events, centrality, lo, hi)',
    '            for k in obs_keys:',
    '                data[k].append(obs[k])',
    '            ns.append(obs["n"])',
    '        result[tag] = dict(mid=mid, bins=bins, n=np.array(ns),',
    '                           **{k: np.array(data[k]) for k in obs_keys})',
    '    return result',
    '',
    'def run_design_point(args):',
    '    """',
    '    Full per-design-point pipeline (called from ProcessPoolExecutor):',
    '      1. build_isobars.py  (N_ISO_PER_JOB cores)',
    '      2. trento_sync       (1 core)',
    '      3. eccentricities    (numpy, in-process)',
    '    args = (idx, R, a, beta3, beta4, w)',
    '    Returns a results dict, raises RuntimeError on hard failure.',
    '    """',
    '    idx, R, a, beta3, beta4, w = args',
    '    t_start = time.perf_counter()',
    '',
    '    design_dir = WORK_DIR / "design_{:04d}".format(idx)',
    '    design_dir.mkdir(parents=True, exist_ok=True)',
    '    nuclei_dir = design_dir / "nuclei"',
    '    nuclei_dir.mkdir(exist_ok=True)',
    '    ws1 = nuclei_dir / "WS1.hdf"',
    '    ws2 = nuclei_dir / "WS2.hdf"',
    '',
    '    # ── Step 1: Isobar Sampler ────────────────────────────────────────────────',
    '    t_iso_start = time.perf_counter()',
    '    if ws1.exists() and ws2.exists() and ws1.stat().st_size > 0:',
    '        t_iso = 0.0',
    '        print("  [{:04d}] isobar cached".format(idx), flush=True)',
    '    else:',
    '        isobar_conf = design_dir / "isobar-conf.yaml"',
    '        write_yaml(isobar_conf, {',
    '            "isobar_samples": {',
    '                "description": "208Pb design {:04d}".format(idx),',
    '                "number_configs":  {"description": "N",   "value": N_CONFIGS},',
    '                "number_nucleons": {"description": "A",   "value": A_PB},',
    '                "seeds_file":      {"description": "HDF",',
    '                                    "filename": str(SEEDS_HDF.resolve())},',
    '                "output_path":     {"description": "dir",',
    '                                    "dirname": str(nuclei_dir.resolve())},',
    '                "number_of_parallel_processes":',
    '                                   {"description": "cores", "value": N_ISO_PER_JOB},',
    '            },',
    '            "isobar_properties": {',
    '                "description": "WS1=projectile WS2=target",',
    '                "isobar1": _isobar_block("WS1", R, a, beta3, beta4),',
    '                "isobar2": _isobar_block("WS2", R, a, beta3, beta4),',
    '            },',
    '        })',
    '        r_iso = subprocess.run(',
    '            [sys.executable,',
    '             str(ISO_DIR / "exec" / "build_isobars.py"),',
    '             str(isobar_conf)],',
    '            cwd=str(ISO_DIR), capture_output=True, text=True,',
    '        )',
    '        if r_iso.returncode != 0:',
    '            raise RuntimeError(',
    '                "[{:04d}] build_isobars failed:\\n{}".format(idx, r_iso.stderr[-500:])',
    '            )',
    '        # Relocate if build_isobars wrote to CWD instead of output_path',
    '        for name in ["WS1.hdf", "WS2.hdf"]:',
    '            for src_path in [nuclei_dir / name, ISO_DIR / name, Path(".") / name]:',
    '                if src_path.exists() and src_path.stat().st_size > 0:',
    '                    dst = nuclei_dir / name',
    '                    if src_path != dst:',
    '                        shutil.move(str(src_path), str(dst))',
    '                    break',
    '        if not (ws1.exists() and ws2.exists()):',
    '            raise FileNotFoundError(',
    '                "[{:04d}] WS1/WS2.hdf not found after build_isobars".format(idx)',
    '            )',
    '        t_iso = time.perf_counter() - t_iso_start',
    '        print("  [{:04d}] isobar done  {}  R={:.3f} a={:.3f} b3={:.4f} b4={:.4f}".format(',
    '              idx, fmt(t_iso), R, a, beta3, beta4), flush=True)',
    '',
    '    # ── Step 2: trento_sync ───────────────────────────────────────────────────',
    '    cache = design_dir / "trento_events.npy"',
    '    t_tr_start = time.perf_counter()',
    '    if cache.exists() and cache.stat().st_size > 0:',
    '        arr  = np.load(cache)',
    '        t_tr = 0.0',
    '        print("  [{:04d}] trento cached  ({:,} events)".format(idx, len(arr)), flush=True)',
    '    else:',
    '        cmd = [',
    '            str(TRENTO_BIN),',
    '            str(ws1), str(ws2),',
    '            str(N_EVENTS),',
    '            "-p", "{:.6f}".format(FIX_P),',
    '            "-k", "{:.6f}".format(FIX_K),',
    '            "-w", "{:.6f}".format(w),',
    '            "-d", "{:.6f}".format(FIX_DMIN),',
    '            "-n", "{:.6f}".format(FIX_NORM),',
    '            "-x", "{:.4f}".format(SIGMA_NN),',
    '            "--b-max",       "{:.1f}".format(B_MAX),',
    '            "--random-seed", str(1000 + idx),',
    '        ]',
    '        r_tr = subprocess.run(cmd, capture_output=True, text=True)',
    '        t_tr = time.perf_counter() - t_tr_start',
    '        if r_tr.returncode != 0:',
    '            raise RuntimeError(',
    '                "[{:04d}] trento_sync failed:\\n{}".format(idx, r_tr.stderr[-400:])',
    '            )',
    '        rows = []',
    '        for line in r_tr.stdout.splitlines():',
    '            line = line.strip()',
    '            if not line or line.startswith("#"):',
    '                continue',
    '            parts = line.split()',
    '            if len(parts) < N_SYNC_COLS:',
    '                continue',
    '            try:',
    '                rows.append([float(x) for x in parts[:N_SYNC_COLS]])',
    '            except ValueError:',
    '                continue',
    '        if not rows:',
    '            raise RuntimeError("[{:04d}] trento_sync: no events parsed".format(idx))',
    '        arr = np.array(rows, dtype=np.float64)',
    '        np.save(cache, arr)',
    '        rate = len(arr) / t_tr if t_tr > 0 else 0',
    '        print("  [{:04d}] trento done  {:,} evt  {}  {:.0f} evt/s  w={:.3f}".format(',
    '              idx, len(arr), fmt(t_tr), rate, w), flush=True)',
    '',
    '    # ── Step 3: Eccentricities ────────────────────────────────────────────────',
    '    cent     = assign_centrality(arr)',
    '    profiles = compute_centrality_profiles(arr, cent)',
    '',
    '    CENT_CLS = [("0-1",0,1),("0-5",0,5),("5-10",5,10),',
    '                ("0-10",0,10),("20-30",20,30),("40-50",40,50)]',
    '    scalar = {}',
    '    for c_label, c_lo, c_hi in CENT_CLS:',
    '        scalar["{}-{}".format(c_lo, c_hi)] = compute_observables(arr, cent, c_lo, c_hi)',
    '',
    '    t_total = time.perf_counter() - t_start',
    '    return {',
    '        "idx":      idx,',
    '        "profiles": profiles,',
    '        "scalar":   scalar,',
    '        "t_iso":    t_iso,',
    '        "t_tr":     t_tr,',
    '        "t_total":  t_total,',
    '        "n_events": len(arr),',
    '    }',
    '',
]

_tmpl = "\n".join(_template_lines)

# Syntax check before writing
ast.parse(_tmpl)  # will raise SyntaxError if template is broken

_tmpl_path = WORK_DIR / "_worker_template.py"
_tmpl_path.write_text(_tmpl)
print(f"Worker template written → {_tmpl_path}  ({len(_tmpl)} chars)")
print(f"Template syntax: OK  ({_tmpl.count(chr(10))} lines)")

import sys as _sys, importlib

# Read the worker template from disk (avoids all escaping issues)
_worker_template_path = WORK_DIR / "_worker_template.py"
WORKER_MODULE         = WORK_DIR / "_pbpb_worker.py"

# Write the template to disk (only the first time)
if not _worker_template_path.exists():
    raise FileNotFoundError(
        f"Worker template not found at {_worker_template_path}.\n"
        "The template was supposed to be written by Cell 6 setup.\n"
        "Please re-run Cell 6."
    )

# Read template and inject all runtime constants
_tmpl = _worker_template_path.read_text()

_filled = (
    _tmpl
    .replace("INJECT_TRENTO_BIN",   str(TRENTO_BIN))
    .replace("INJECT_ISO_DIR",      str(ISO_DIR))
    .replace("INJECT_WORK_DIR",     str(WORK_DIR))
    .replace("INJECT_SEEDS_HDF",    str(seeds_hdf))
    .replace("INJECT_N_CONFIGS",    str(N_CONFIGS))
    .replace("INJECT_N_EVENTS",     str(N_EVENTS))
    .replace("INJECT_N_ISO_PER_JOB",str(N_ISO_PER_JOB))
    .replace("INJECT_A_PB",         str(A_PB))
    .replace("INJECT_FIX_P",        str(FIX_P))
    .replace("INJECT_FIX_K",        f"{FIX_K:.10f}")
    .replace("INJECT_FIX_NORM",     str(FIX_NORM))
    .replace("INJECT_FIX_DMIN",     f"{FIX_DMIN:.10f}")
    .replace("INJECT_SIGMA_NN",     str(SIGMA_NN))
    .replace("INJECT_B_MAX",        str(B_MAX))
    .replace("INJECT_FIX_BETA2",    str(FIX_BETA2))
    .replace("INJECT_FIX_GAMMA",    str(FIX_GAMMA))
    .replace("INJECT_FIX_CORR_L",   str(FIX_CORR_L))
    .replace("INJECT_FIX_CORR_S",   str(FIX_CORR_S))
    .replace("INJECT_N_SYNC_COLS",  str(N_SYNC_COLS))
    .replace("INJECT_COL_MULT",     str(COL_MULT))
    .replace("INJECT_COL_E2",       str(COL_E2))
    .replace("INJECT_COL_E3",       str(COL_E3))
    .replace("INJECT_COL_E4",       str(COL_E4))
)

WORKER_MODULE.write_text(_filled)
print(f"Worker module written → {WORKER_MODULE}")

# Add WORK_DIR to sys.path so the worker is importable by ProcessPoolExecutor
if str(WORK_DIR) not in _sys.path:
    _sys.path.insert(0, str(WORK_DIR))

# Reload to verify syntax and constants
if "_pbpb_worker" in _sys.modules:
    del _sys.modules["_pbpb_worker"]
import _pbpb_worker as _w

print(f"  TRENTO_BIN = {_w.TRENTO_BIN}")
print(f"  ISO_DIR    = {_w.ISO_DIR}")
print(f"  N_EVENTS   = {_w.N_EVENTS:,}")
print(f"  FIX_K      = {_w.FIX_K:.6f}  (sigma_k=0.97 → k=1/0.97²)")
print(f"  FIX_DMIN   = {_w.FIX_DMIN:.6f} fm  (d³=0.52 fm³)")
print("Worker module OK ✓")


---
## Cell 7 — Parallel Main Loop

Runs all 50 design points using `ProcessPoolExecutor`.
Each worker executes the full pipeline for one design point:
1. `build_isobars.py` (`N_ISO_PER_JOB` cores)
2. `trento_sync` (1 core)
3. Eccentricity cumulants (numpy, in-process)

Already-cached design points are loaded directly — **resume-safe**.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import platform
import _pbpb_worker as _w

print("=" * 60)
print(f"  Stage 5 — Parallel loop  ({N_DESIGN} pts, {N_OUTER_WORKERS} workers)")
print("=" * 60)
print(f"  Parallelism: {N_OUTER_WORKERS} outer workers")
print(f"  Each worker: {N_ISO_PER_JOB} iso cores + 1 trento core")
print(f"  Total CPUs : ~{N_OUTER_WORKERS*(N_ISO_PER_JOB+1)} / {N_CORES}")
print()

# Build argument list: (idx, R, a, beta3, beta4, w)
args_list = [
    (int(i),
     float(design[i,0]), float(design[i,1]),
     float(design[i,2]), float(design[i,3]),
     float(design[i,4]))
    for i in range(N_DESIGN)
]

def _is_cached(idx):
    d  = WORK_DIR / f"design_{idx:04d}"
    ws = d / "nuclei" / "WS1.hdf"
    tr = d / "trento_events.npy"
    return (ws.exists() and ws.stat().st_size > 0
            and tr.exists() and tr.stat().st_size > 0)

cached_idx  = [a[0] for a in args_list if _is_cached(a[0])]
pending     = [a for a in args_list if not _is_cached(a[0])]

print(f"  Cached : {len(cached_idx)} / {N_DESIGN}  (will skip)")
print(f"  To run : {len(pending)}")
print()

all_results = {}

# Load cached points directly
for idx in cached_idx:
    arr  = np.load(WORK_DIR / f"design_{idx:04d}" / "trento_events.npy")
    cent = assign_centrality(arr)
    prof = compute_centrality_profiles(arr, cent)
    scal = {}
    for c_label, c_lo, c_hi in CENT_CLASSES:
        scal[f"{c_lo}-{c_hi}"] = compute_observables(arr, cent, c_lo, c_hi)
    all_results[idx] = dict(idx=idx, profiles=prof, scalar=scal,
                            t_iso=0.0, t_tr=0.0, t_total=0.0, n_events=len(arr))

# Run pending points in parallel
t_wall_start = time.perf_counter()
timing       = []

if pending:
    # Use fork on Linux (faster, shares memory); spawn on macOS
    ctx = __import__("multiprocessing").get_context(
        "fork" if platform.system() == "Linux" else "spawn"
    )
    with ProcessPoolExecutor(max_workers=N_OUTER_WORKERS,
                             mp_context=ctx) as executor:
        futures = {
            executor.submit(_w.run_design_point, a): a[0]
            for a in pending
        }
        n_done = 0
        for future in as_completed(futures):
            idx_done = futures[future]
            n_done  += 1
            try:
                res = future.result()
            except Exception as exc:
                print(f"  [{idx_done:04d}] FAILED: {exc}", flush=True)
                continue

            all_results[res["idx"]] = res
            if res["t_tr"] > 0:
                timing.append(res["t_total"])

            wall   = time.perf_counter() - t_wall_start
            avg    = float(np.mean(timing)) if timing else 0.0
            n_left = len(pending) - n_done
            eta    = avg * n_left / N_OUTER_WORKERS
            print(f"  Progress {n_done:3d}/{len(pending)}  "
                  f"wall={fmt(wall)}  avg/pt={fmt(avg)}  "
                  f"ETA={fmt(eta)}", flush=True)

t_wall = time.perf_counter() - t_wall_start
n_ok   = len(all_results)
failed = [i for i in range(N_DESIGN) if i not in all_results]

print()
print("=" * 60)
print("  TIMING REPORT")
print("=" * 60)
print(f"  Design pts done    : {n_ok} / {N_DESIGN}")
if failed:
    print(f"  FAILED pts         : {failed}")
print(f"  Total wall time    : {fmt(t_wall)}")
if timing:
    print(f"  Mean time/pt       : {fmt(float(np.mean(timing)))}")
    print(f"  Min  time/pt       : {fmt(float(np.min(timing)))}")
    print(f"  Max  time/pt       : {fmt(float(np.max(timing)))}")
    ideal    = float(np.mean(timing)) * len(pending) / t_wall if t_wall > 0 else 0
    print(f"  Parallel speedup   : {ideal:.1f}×  (ideal = {N_OUTER_WORKERS}×)")
    print()
    t_seq    = float(np.mean(timing)) * N_DESIGN
    t_par    = t_seq / N_OUTER_WORKERS
    print(f"  Budget ({N_DESIGN} pts, {N_EVENTS:,} events/pt):")
    print(f"    Sequential (1 worker) : {fmt(t_seq)}")
    print(f"    Parallel ({N_OUTER_WORKERS} workers)  : {fmt(t_par)}")
    print(f"    (on {N_CORES}-core machine)")


---
## Cell 8 — Save Results

In [ ]:
print("\n=== Saving eccentricities ===")

save_dict = dict(
    design=design, unit_lhs=unit_lhs,
    param_names=np.array(PNAMES), param_lo=PLO, param_hi=PHI,
    obs_keys=np.array(OBS_KEYS),
)

for c_label, c_lo, c_hi in CENT_CLASSES:
    c_key = f"{c_lo}-{c_hi}"
    for ok in OBS_KEYS:
        save_dict[f"{ok}_c{c_key}"] = np.array([
            all_results[i]["scalar"].get(c_key, {}).get(ok, np.nan)
            if i in all_results else np.nan
            for i in range(N_DESIGN)
        ])

for tag in ["uc", "wide"]:
    valid = [all_results[i]["profiles"] for i in range(N_DESIGN)
             if i in all_results and all_results[i].get("profiles")]
    if valid:
        mid = valid[0][tag]["mid"]
        save_dict[f"cent_mid_{tag}"] = mid
        for ok in OBS_KEYS:
            save_dict[f"{ok}_{tag}"] = np.array([
                all_results[i]["profiles"][tag][ok]
                if i in all_results and all_results[i].get("profiles")
                else np.full(len(mid), np.nan)
                for i in range(N_DESIGN)
            ])

out_npz = WORK_DIR / "eccentricities.npz"
np.savez_compressed(out_npz, **save_dict)
print(f"Saved → {out_npz}")
print(f"Keys  : {list(save_dict.keys())[:6]} …")


---
## Cell 9 — Plots

In [ ]:
print("\n=== Plots ===")
plots_dir = WORK_DIR / "plots"
plots_dir.mkdir(exist_ok=True)

BG,PANEL,GRID,TEXT,TICK = "#0d0f14","#13161e","#1e2230","#d8dce8","#8892a4"
PCOL = ["#4fc3f7","#ffd54f","#ff4081","#b39ddb","#80cbc4"]

def _dark():
    plt.rcParams.update({
        "figure.facecolor":BG,"axes.facecolor":PANEL,
        "axes.edgecolor":GRID,"axes.labelcolor":TEXT,
        "xtick.color":TICK,"ytick.color":TICK,
        "text.color":TEXT,"grid.color":GRID,
        "grid.linewidth":0.4,"grid.alpha":0.5,"font.size":9,
        "legend.facecolor":PANEL,"legend.edgecolor":GRID,
        "legend.labelcolor":TEXT,"legend.fontsize":7.5,
    })

OBS_META = {
    "e2_2"    :(r"$\varepsilon_2\{2\}$",    "#4fc3f7"),
    "e3_2"    :(r"$\varepsilon_3\{2\}$",    "#ff4081"),
    "e4_2"    :(r"$\varepsilon_4\{2\}$",    "#ffd54f"),
    "T32"     :(r"$\varepsilon_3\{2\}/\varepsilon_2\{2\}$","#b39ddb"),
    "R42"     :(r"$\varepsilon_2\{4\}/\varepsilon_2\{2\}$","#80cbc4"),
    "e3_ratio":(r"$\varepsilon_3\{2\}/\varepsilon_3\{4\}$","#ffcc80"),
    "e4_4c"   :(r"$\varepsilon_4\{4\}^4$",  "#f48fb1"),
}

valid_prof = [all_results[i]["profiles"] for i in range(N_DESIGN)
              if i in all_results and all_results[i].get("profiles")]
obs_list   = list(OBS_META.keys())

# ── LHS corner ────────────────────────────────────────────────────────────────
_dark()
n = N_PAR
fig, axes = plt.subplots(n, n, figsize=(n*2.6, n*2.6))
fig.patch.set_facecolor(BG)
fig.suptitle(
    r"$^{208}$Pb+$^{208}$Pb Deformed — LHS Design"
    f"\n{N_DESIGN} pts × {N_PAR} params  "
    r"($\sqrt{s_{NN}}=2.76$ TeV)",
    fontsize=10, color=TEXT, y=1.01, fontweight="bold")
for row in range(n):
    for col in range(n):
        ax = axes[row, col]
        ax.set_facecolor(PANEL)
        for sp in ax.spines.values():
            sp.set_edgecolor(GRID); sp.set_linewidth(0.5)
        if row == col:
            ax.hist(unit_lhs[:, col], bins=12, range=(0,1),
                    color=PCOL[col], alpha=0.75, edgecolor=BG, lw=0.3)
            ax.set_xlim(0,1); ax.set_yticks([])
            if row == n-1:
                lo, hi = PLO[col], PHI[col]
                ax.set_xticks([0,.5,1])
                ax.set_xticklabels([f"{lo:.2g}",f"{(lo+hi)/2:.3g}",f"{hi:.2g}"],
                                   fontsize=7, color=TICK)
            else: ax.set_xticks([])
        elif row > col:
            ax.scatter(unit_lhs[:,col], unit_lhs[:,row],
                       c=PCOL[col], s=8, alpha=0.5, linewidths=0)
            ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05)
            for v in [.25,.5,.75]:
                ax.axhline(v,color=GRID,lw=0.3,ls="--",alpha=0.5)
                ax.axvline(v,color=GRID,lw=0.3,ls="--",alpha=0.5)
            if row == n-1:
                lo,hi=PLO[col],PHI[col]; ax.set_xticks([0,.5,1])
                ax.set_xticklabels([f"{lo:.2g}",f"{(lo+hi)/2:.3g}",f"{hi:.2g}"],
                                   fontsize=7,color=TICK)
            else: ax.set_xticks([])
            if col == 0:
                lo,hi=PLO[row],PHI[row]; ax.set_yticks([0,.5,1])
                ax.set_yticklabels([f"{lo:.2g}",f"{(lo+hi)/2:.3g}",f"{hi:.2g}"],
                                   fontsize=7,color=TICK)
            else: ax.set_yticks([])
        else: ax.set_visible(False)
for j in range(n):
    axes[n-1,j].set_xlabel(PLABELS[j],fontsize=9,color=PCOL[j],labelpad=5)
    if j>0: axes[j,0].set_ylabel(PLABELS[j],fontsize=9,color=PCOL[j],labelpad=5)
fig.tight_layout(h_pad=0.15, w_pad=0.15)
p = plots_dir / "lhs_corner.pdf"
fig.savefig(p, dpi=150, bbox_inches="tight", facecolor=BG)
plt.close(fig)
print(f"  Saved: {p}")

# ── Eccentricities vs centrality ──────────────────────────────────────────────
if valid_prof:
    _dark()
    fig, axes = plt.subplots(len(obs_list), 2,
                             figsize=(13, len(obs_list)*2.6), squeeze=False)
    fig.patch.set_facecolor(BG)
    fig.suptitle(
        r"$^{208}$Pb+$^{208}$Pb Deformed — Eccentricities vs Centrality"
        f"\n{N_DESIGN} pts | {N_EVENTS:,} evt/pt | trento_sync | β₃,β₄ free",
        fontsize=10, color=TEXT, y=1.005, fontweight="bold")
    for row, key in enumerate(obs_list):
        label, color = OBS_META[key]
        for col, (tag, xlim) in enumerate([("uc",(0,10)),("wide",(0,90))]):
            ax = axes[row,col]; ax.set_facecolor(PANEL)
            for sp in ax.spines.values():
                sp.set_edgecolor(GRID); sp.set_linewidth(0.5)
            ax.grid(True, ls="--", alpha=0.35)
            curves = []
            for prof in valid_prof:
                y   = prof[tag][key]
                mid = prof[tag]["mid"]
                ok  = np.isfinite(y)
                if ok.sum() < 2: continue
                ax.plot(mid[ok], y[ok], color=color, alpha=0.12, lw=0.7)
                curves.append(y)
            if curves:
                mat  = np.array(curves)
                xm   = valid_prof[0][tag]["mid"]
                med  = np.nanmedian(mat, axis=0)
                lo16 = np.nanpercentile(mat, 16, axis=0)
                hi84 = np.nanpercentile(mat, 84, axis=0)
                ok   = np.isfinite(med)
                ax.plot(xm[ok], med[ok], color=color, lw=2.0, label="median")
                ax.fill_between(xm[ok], lo16[ok], hi84[ok],
                                color=color, alpha=0.25, label="16–84%")
            ax.set_ylabel(label, fontsize=9, color=color)
            ax.set_xlabel(r"Centrality $c$ (%)", fontsize=8)
            ax.set_xlim(*xlim)
            ax.set_title("UC 1% bins" if col==0 else "Wide 0–90%",
                         fontsize=8, color=TICK, pad=3)
            if row == 0: ax.legend(loc="upper right", fontsize=7)
    fig.tight_layout(h_pad=0.3, w_pad=0.5)
    p = plots_dir / "ecc_vs_centrality.pdf"
    fig.savefig(p, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    print(f"  Saved: {p}")

# ── Eccentricities vs parameters ──────────────────────────────────────────────
for c_label, c_lo, c_hi in [("0-1%",0,1),("0-5%",0,5),("20-30%",20,30)]:
    c_key = f"{c_lo}-{c_hi}"
    _dark()
    fig, axes = plt.subplots(len(obs_list), N_PAR,
                             figsize=(N_PAR*2.8, len(obs_list)*2.4), squeeze=False)
    fig.patch.set_facecolor(BG)
    fig.suptitle(
        r"$^{208}$Pb+$^{208}$Pb Deformed — Ecc vs Params  "
        f"(cent {c_label})\n4 Isobar-Sampler params + TRENTo w",
        fontsize=10, color=TEXT, y=1.004, fontweight="bold")
    for r_obs, okey in enumerate(obs_list):
        obs_label, color = OBS_META[okey]
        for c_par in range(N_PAR):
            ax = axes[r_obs, c_par]; ax.set_facecolor(PANEL)
            for sp in ax.spines.values():
                sp.set_edgecolor(GRID); sp.set_linewidth(0.5)
            ax.grid(True, ls="--", alpha=0.3)
            xs, ys = [], []
            for i in range(N_DESIGN):
                if i not in all_results: continue
                y = all_results[i]["scalar"].get(c_key, {}).get(okey, np.nan)
                if not np.isfinite(y): continue
                xs.append(design[i, c_par]); ys.append(y)
            if len(xs) > 3:
                xa, ya = np.array(xs), np.array(ys)
                try:
                    dens = gaussian_kde(np.vstack([xa,ya]))(np.vstack([xa,ya]))
                    srt  = np.argsort(dens)
                    ax.scatter(xa[srt], ya[srt], c=dens[srt], cmap="plasma",
                               s=14, linewidths=0, alpha=0.85)
                except Exception:
                    ax.scatter(xa, ya, color=color, s=10, alpha=0.6)
                if len(xa) >= 10:
                    order = np.argsort(xa)
                    xs_s, ys_s = xa[order], ya[order]
                    ws_ = max(3, len(xs_s) // 8)
                    xsm, ysm = [], []
                    for st in range(0, len(xs_s)-ws_, ws_//2):
                        xsm.append(np.median(xs_s[st:st+ws_]))
                        ysm.append(np.median(ys_s[st:st+ws_]))
                    ax.plot(xsm, ysm, color="white", lw=1.2, alpha=0.7)
            if c_par == 0:   ax.set_ylabel(obs_label, fontsize=9, color=color)
            if r_obs == len(obs_list)-1: ax.set_xlabel(PLABELS[c_par], fontsize=8.5)
            if r_obs == 0:   ax.set_title(PLABELS[c_par], fontsize=8.5,
                                          color=TEXT, pad=3)
    fig.tight_layout(h_pad=0.3, w_pad=0.4)
    safe = c_label.replace("%","pct").replace("-","_")
    p = plots_dir / f"ecc_vs_params_{safe}.pdf"
    fig.savefig(p, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    print(f"  Saved: {p}")

print(f"\nAll outputs in: {WORK_DIR.resolve()}")
print(f"  lhs_design.npz")
print(f"  eccentricities.npz")
print(f"  design_NNNN/nuclei/WS1.hdf + WS2.hdf")
print(f"  design_NNNN/trento_events.npy")
print(f"  plots/lhs_corner.pdf")
print(f"  plots/ecc_vs_centrality.pdf")
print(f"  plots/ecc_vs_params_0_1pct.pdf")
print(f"  plots/ecc_vs_params_0_5pct.pdf")
print(f"  plots/ecc_vs_params_20_30pct.pdf")
